# 用三国演义的数据
* data = three_kingdoms.txt


## 1. 数据处理以及分词
* 依赖：jieba

In [1]:
# 对txt文件进行中文分词
import jieba
import os
from utils import files_processing

# 源文件所在目录
os.makedirs('./three_kingdoms/segment', exist_ok=True)  # exist_ok=True：文件夹存在也不会报错

source_folder = './three_kingdoms/source'
segment_folder = './three_kingdoms/segment'

# 字词分割，对整个文件内容进行字词分割
def segment_lines(file_list,segment_out_dir,stopwords=[]):
    for i,file in enumerate(file_list):
        segment_out_name=os.path.join(segment_out_dir,'segment_{}.txt'.format(i))
        with open(file, 'rb') as f:
            document = f.read()
            document_cut = jieba.cut(document)
            sentence_segment=[]
            for word in document_cut:
                if word not in stopwords:
                    sentence_segment.append(word)
            result = ' '.join(sentence_segment)
            result = result.encode('utf-8')
            with open(segment_out_name, 'wb') as f2:
                f2.write(result)

# 对source中的txt文件进行分词，输出到segment目录中
file_list=files_processing.get_files_list(source_folder, postfix='*.txt')
segment_lines(file_list, segment_folder)

Building prefix dict from the default dictionary ...
Loading model from cache C:\Users\SomeH\AppData\Local\Temp\jieba.cache
Loading model cost 0.841 seconds.
Prefix dict has been built successfully.


## 2. 分词转向量
* word2vec.Word2Vec

In [10]:
# 先运行 word_seg进行中文分词，然后再进行word_similarity计算
# 将Word转换成Vec，然后计算相似度 
from gensim.models import word2vec
import multiprocessing
import os

# 如果目录中有多个文件，可以使用PathLineSentences
segment_folder = './three_kingdoms/segment'
# 切分之后的句子合集
sentences = word2vec.PathLineSentences(segment_folder)

# 设置模型参数，进行训练
model = word2vec.Word2Vec(sentences, vector_size=100, window=3, min_count=1)
# print(model.wv['曹操'])
print("和曹操最相近的词：")
print(sorted(model.wv.most_similar(positive=['曹操'], negative=[]), key=lambda x: x[1], reverse=True))
print("曹操+刘备-张飞=")
print(sorted(model.wv.most_similar(positive=['曹操', '刘备'], negative=['张飞']), key=lambda x: x[1], reverse=True))
# 设置模型参数，进行训练
model2 = word2vec.Word2Vec(sentences, vector_size=128, window=5, min_count=5, workers=multiprocessing.cpu_count())
# 保存模型
# ==========新增这两行==========
save_dir = "./three_kingdoms/models"
os.makedirs(save_dir, exist_ok=True)  # exist_ok=True：文件夹存在也不会报错
model2.save('./three_kingdoms/models/word2Vec.model')
print("=================模型2====================")
print("和曹操最相近的词：")
print(sorted(model2.wv.most_similar(positive=['曹操'], negative=[]), key=lambda x: x[1], reverse=True))
print("曹操+刘备-张飞=")
print(sorted(model2.wv.most_similar(positive=['曹操', '刘备'], negative=['张飞']), key=lambda x: x[1], reverse=True))

和曹操最相近的词：
[('关公', 0.9921773076057434), ('孙权', 0.9907181859016418), ('孔明', 0.9891878366470337), ('周瑜', 0.9880921244621277), ('先主', 0.9878199100494385), ('云长', 0.98619544506073), ('司马懿', 0.9853874444961548), ('众将', 0.9852510690689087), ('又', 0.985184371471405), ('夫人', 0.9846261739730835)]
曹操+刘备-张飞=
[('丞相', 0.9911447763442993), ('某', 0.9910182356834412), ('今', 0.9884661436080933), ('将军', 0.9878004193305969), ('臣', 0.9876855611801147), ('卿', 0.9874168634414673), ('既', 0.986638605594635), ('主公', 0.9853851199150085), ('吾', 0.9841712117195129), ('大丈夫', 0.9839144945144653)]
=================模型2====================
和曹操最相近的词：
[('孙权', 0.9660531878471375), ('众将', 0.9589381814002991), ('关公', 0.9567785859107971), ('飞', 0.9502373933792114), ('先主', 0.9502055048942566), ('惊', 0.9498162865638733), ('良久', 0.948656439781189), ('关将军', 0.9483081698417664), ('乃谓', 0.947752058506012), ('姓名', 0.9464918971061707)]
曹操+刘备-张飞=
[('今', 0.9735897183418274), ('吾', 0.9726005792617798), ('丞相', 0.963232696056366), ('臣', 